In [1]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [2]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [3]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:03<00:00, 4403074.20it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [4]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


In [5]:
protein_interaction[protein_interaction['Translated_protein_2'] == '']

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2


In [6]:
# ppi_available_genes = list(set(protein_interaction['Translated_protein_1']).union(set(protein_interaction['Translated_protein_2'])))

In [7]:
# len(set(ppi_available_genes))

In [8]:
# protein_working = protein_interaction[protein_interaction['combined_score']>700]
# protein_working[protein_working['Translated_protein_1'] == 'EGFR'][protein_interaction['Translated_protein_2'] =='SOX2']

### DrugBank

In [9]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [10]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        
        # Extract toxicity information at drug level
        toxicity = drug.find('db:toxicity', ns)
        toxicity_text = toxicity.text if toxicity is not None else None
        
        # Extract categories (may contain toxicity classifications)
        categories = drug.findall('db:categories/db:category', ns)
        category_list = []
        for cat in categories:
            cat_name = cat.find('db:category', ns)
            if cat_name is not None:
                category_list.append(cat_name.text)
        categories_str = ', '.join(category_list) if category_list else None
        
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None,
                    'toxicity': toxicity_text,
                    'categories': categories_str
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None,
                        'toxicity': toxicity_text,
                        'categories': categories_str
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [11]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Genetic results

In [12]:
### import data
# ### genes
# hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
# hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

# hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
# hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
import pandas as pd
### genes
### import data
### genes
hpv_positive_amplification_top_gene_df= pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV positive amplification top genes.csv')
hpv_positive_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_positive_deletion_top_gene_df= pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV positive deletion top genes.csv')
hpv_positive_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_positive_deletion_top_gene_df['amplification_count'] = hpv_positive_deletion_top_gene_df['deletion_count']
hpv_positive_genes = pd.concat([hpv_positive_amplification_top_gene_df, hpv_positive_deletion_top_gene_df], axis=0)
hpv_positive_genes =hpv_positive_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

hpv_negative_amplification_top_gene_df = pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV negative amplification top genes.csv')
hpv_negative_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_deletion_top_gene_df = pd.read_csv('../2. Genetic based drug repurposing/Results/CNV results/HPV negative deletion top genes.csv')
hpv_negative_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_negative_deletion_top_gene_df['amplification_count'] = hpv_negative_deletion_top_gene_df['deletion_count']
hpv_negative_genes = pd.concat([hpv_negative_amplification_top_gene_df, hpv_negative_deletion_top_gene_df], axis=0)
hpv_negative_genes =hpv_negative_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('../2. Genetic based drug repurposing/Results/SOM results/HPV positive top genes.csv')
hpv_negative_som_genes = pd.read_csv('../2. Genetic based drug repurposing/Results/SOM results/HPV negative top genes.csv')


### Literature results

In [13]:
extracted_target_df= pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
scraped_lit_data = pd.read_csv('../3. Literature based validation/Data/extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv', on_bad_lines='skip')
extracted_target_df_combined = pd.read_csv('../3. Literature based validation/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined['GENE'] = extracted_target_df_combined['GENE'].str.upper()

In [14]:
key_genes = ['PIK3CA', 'TP53', 'CDKN2A','SOX2', 'EGFR', 'HRAS', 'NOTCH1', 'FAT1', 'CASP8', 'KMT2D']
key_genes = [gene.lower() for gene in key_genes]
extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]
key_genes_indexes = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)].index.tolist()
key_genes_PMID = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]['PMID'].unique().tolist()

In [15]:
key_genes = ['SOX2', 'TP53']
key_genes = [gene.lower() for gene in key_genes]
extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]
key_genes_indexes = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)].index.tolist()
key_genes_PMID = extracted_target_df[extracted_target_df['GENE'].isin(key_genes)]['PMID'].unique().tolist()

In [16]:
scraped_lit_data[scraped_lit_data['PMID'].isin(key_genes_PMID)]

,Unnamed: 0.1,Unnamed: 0,PMID,TITLE,YEAR,ABSTRACT,TARGETS,SENTENCE_WHERE_FOUND
3533,131671,131671,11390535,Detecting colorectal cancer in stool with the ...,2001.0,BACKGROUND:Colorectal cancer cells are shed in...,"tp53, bat26, k-ras",results showed that tp53 gene mutations were d...
4260,132398,132398,11445847,The relationship between allelic imbalance on ...,2001.0,The huge majority of head and neck squamous ce...,tp53,"""a nuclear accumulation of p53 (51%) was indep..."
4261,132399,132399,11445859,Microsatellite and chromosome instability in s...,2001.0,Head-and neck squamous cell carcinoma (HNSCC) ...,"3p, 8q, 17q, tp53","""in a group of 20 patients loh was observed ma..."
8817,136963,136963,11916556,Radiobiological characteristics of solid tumou...,2002.0,Human head and neck squamous cell carcinoma ce...,tp53,"""in both total and q cell fractions, sas/mtp53..."
15054,143211,143211,12589859,Association of chromium exposure with multiple...,2003.0,A 56-year-old man who had worked at a chromate...,"d2s123, d3s1067, tp53, d18s474",the present case seems to have resulted from l...
15944,144101,144101,12673364,TP53 and mutations in human cancer.,2003.0,TP53 is the most frequently mutated gene in hu...,"tp53, ser-249 tp53 mutation","in several other cancers, such as squamous cel..."
15954,144111,144111,12673679,"TP53, p14ARF, p16INK4a and H-ras gene molecula...",2003.0,Intestinal-type adenocarcinoma (ITAC) of the n...,"tp53, p14(arf) and p16(ink4a)","""in patients with known exposure, cumulative e..."
16231,144389,144389,12702551,A genetic explanation of Slaughter's concept o...,2003.0,"The concept of ""field cancerization"" was first...","tp53, field cancerization",the development of an expanding preneoplastic ...
16319,144477,144477,12708486,Usefulness of combined treatment with mild tem...,2003.0,Human head and neck squamous cell carcinoma ce...,"tp53, p53","""the tumor cell suspensions thus obtained were..."
18815,146975,146975,12935924,The spectrum of mutations in TP53 in laryngeal...,2003.0,Laryngeal cancer (LC) is the sixth most common...,"tp53, exons 5-8","""in tumor-derived samples, mutations were foun..."


In [17]:
gene_inv = ['HDAC2', 'DPP4', 'ACE2']
extracted_target_df_combined[extracted_target_df_combined['GENE'].isin(gene_inv)]

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
1860,DPP4,"11752453, 15735049, 16319515, 16817140, 170914...","6791, 34785, 40958, 45901, 48671, 58966",6


In [18]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

### Functions

In [19]:
# temp = extracted_target_df_combined.copy()

# ### check that all were capitilized in the first place
# temp[temp['GENE'] != extracted_target_df_combined['GENE']]

In [20]:
def connect_to_genes(gene_df, ppi = protein_interaction, literature_data = extracted_target_df_combined):
    ### case fix
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    ppi['Translated_protein_1'] = ppi['Translated_protein_1'].str.upper()
    ppi['Translated_protein_2'] = ppi['Translated_protein_2'].str.upper()
    literature_data['GENE'] = literature_data['GENE'].str.upper()

    ### connect genes to those in PPI
    ### ensure only strong interactions are considered
    ppi = ppi[ppi['combined_score']>=700].reset_index(drop=True)
    ppi_available_genes = set(ppi['Translated_protein_1']).union(set(ppi['Translated_protein_2']))
    #### get literature validated genes
    literature_validated_genes = literature_data['GENE'].values
    for i,gene in tqdm(enumerate(gene_df['gene_name'])):
        validated_connected_genes = []
        PMIDs_connected_genes = []
        connected_genes = []
        if gene in ppi_available_genes:
            ### get all connected genes
            connected_genes = ppi[ppi['Translated_protein_1'] == gene]['Translated_protein_2'].tolist()
            connected_genes += ppi[ppi['Translated_protein_2'] == gene]['Translated_protein_1'].tolist()
            connected_genes = list(set(connected_genes))
            #### check which of the connected genes are literature validated
            for gene2 in connected_genes:
                if gene2 in literature_validated_genes:
                    validated_connected_genes.append(gene2)
                    ### get PMIDs for the gene
                    pmids = literature_data[literature_data['GENE'] == gene2]['PMID'].tolist()
                    PMIDs_connected_genes += pmids
        else:
            connected_genes = []
            validated_connected_genes = []
        gene_df.loc[i, 'literature_validated_connected_genes'] = ', '.join(validated_connected_genes)
        gene_df.loc[i, 'num_literature_validated_connected_genes'] = len(validated_connected_genes)
        gene_df.loc[i, 'PMIDs_connected_genes'] = ', '.join(map(str, set(PMIDs_connected_genes)))

    return gene_df

def connect_to_genes_explicit(gene_df, ppi=protein_interaction, literature_data=extracted_target_df_combined):
    """
    Expanded explicit version: For each risk gene, create a row for each neighbor (not grouped), with all details from grouped version.
    Returns a DataFrame with columns:
        risk_gene, MUT_TYPE, q_value, empirical_q_value, PMID, NUMBER_OF_ARTICLES,
        neighbor_gene, neighbor_is_lit_validated, neighbor_PMID, neighbor_NUMBER_OF_ARTICLES,
        neighbor_is_lit_validated, neighbor_PMIDs, neighbor_NUMBER_OF_ARTICLES,
    """
    ### case correction for making connections
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    ppi['Translated_protein_1'] = ppi['Translated_protein_1'].str.upper()
    ppi['Translated_protein_2'] = ppi['Translated_protein_2'].str.upper()
    literature_data['GENE'] = literature_data['GENE'].str.upper()

    ppi = ppi[ppi['combined_score'] >= 700].reset_index(drop=True)
    ppi_available_genes = set(ppi['Translated_protein_1']).union(set(ppi['Translated_protein_2']))
    literature_validated_genes = set(literature_data['GENE'].values)
    # Prepare lookup for literature data
    lit_lookup = literature_data.set_index('GENE')
    rows = []
    for idx, risk_row in gene_df.iterrows():
        risk_gene = risk_row['gene_name']
        mut_type = risk_row.get('MUT_TYPE', None)
        q_value = risk_row.get('q_value', None)
        empirical_q_value = risk_row.get('empirical_q_value', None)
        PMID = risk_row.get('PMID', None)
        NUMBER_OF_ARTICLES = risk_row.get('NUMBER_OF_ARTICLES', None)
        if risk_gene in ppi_available_genes:
            neighbors = ppi[ppi['Translated_protein_1'] == risk_gene]['Translated_protein_2'].tolist()
            neighbors += ppi[ppi['Translated_protein_2'] == risk_gene]['Translated_protein_1'].tolist()
            neighbors = list(set(neighbors))
            for neighbor in neighbors:
                neighbor_is_lit_validated = neighbor in literature_validated_genes
                neighbor_PMIDs = ''
                neighbor_NUMBER_OF_ARTICLES = None
                neighbor_PMID = None
                if neighbor_is_lit_validated and neighbor in lit_lookup.index:
                    neighbor_PMIDs = ', '.join(map(str, lit_lookup.loc[neighbor]['PMID'])) if isinstance(lit_lookup.loc[neighbor]['PMID'], (list, pd.Series)) else str(lit_lookup.loc[neighbor]['PMID'])
                    neighbor_NUMBER_OF_ARTICLES = lit_lookup.loc[neighbor]['NUMBER_OF_ARTICLES']
                    neighbor_PMID = lit_lookup.loc[neighbor]['PMID']
                rows.append({
                    'risk_gene': risk_gene,
                    'MUT_TYPE': mut_type,
                    'q_value': q_value,
                    'empirical_q_value': empirical_q_value,
                    'PMID': PMID,
                    'NUMBER_OF_ARTICLES': NUMBER_OF_ARTICLES,
                    'neighbor_gene': neighbor,
                    'neighbor_is_lit_validated': neighbor_is_lit_validated,
                    'neighbor_PMID': neighbor_PMID,
                    'neighbor_NUMBER_OF_ARTICLES': neighbor_NUMBER_OF_ARTICLES,
                    'neighbor_PMIDs': neighbor_PMIDs
                })
        else:
            rows.append({
                'risk_gene': risk_gene,
                'MUT_TYPE': mut_type,
                'q_value': q_value,
                'empirical_q_value': empirical_q_value,
                'PMID': PMID,
                'NUMBER_OF_ARTICLES': NUMBER_OF_ARTICLES,
                'neighbor_gene': None,
                'neighbor_is_lit_validated': False,
                'neighbor_PMID': None,
                'neighbor_NUMBER_OF_ARTICLES': None,
                'neighbor_PMIDs': ''
            })
    expanded_df = pd.DataFrame(rows)
    return expanded_df

## HPV+

#### Genes

In [21]:
hpv_positive_som_genes

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,PIK3CA,18,17,0.005473,0.005169,4.738394e-19,2.079207e-15,True,0.0001,0.036563,23.611111,0.129219
1,ZNF750,11,8,0.005071,0.003688,7.324983e-12,1.607101e-08,True,0.0001,0.036563,11.111111,0.056350
2,CYLD,8,8,0.002677,0.002677,6.578872e-07,9.622697e-04,True,0.0001,0.036563,11.111111,0.029749
3,EP300,9,9,0.001243,0.001243,6.021135e-05,4.723959e-02,True,0.0001,0.036563,12.500000,0.015534
4,CCDC191,6,6,0.002083,0.002083,6.589348e-05,4.723959e-02,True,0.0001,0.036563,8.333333,0.017355
5,LRRC37B,6,5,0.001994,0.001662,8.343138e-05,4.723959e-02,True,0.0001,0.036563,6.944444,0.013847


In [22]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
hpv_positive_combined_genes

,gene_name,MUT_TYPE,q_value,empirical_q_value
0,AC003035.1,DELETION,8.234936784924659e-31,0.0203434456037747
1,AC003035.2,DELETION,8.234936784924659e-31,0.0203434456037747
2,AC003657.1,DELETION,8.234936784924659e-31,0.0203434456037747
3,AC003666.1,DELETION,8.234936784924659e-31,0.0203434456037747
4,AC003669.1,DELETION,8.234936784924659e-31,0.0203434456037747
...,...,...,...,...
442,ZFX-AS1,DELETION,8.234936784924659e-31,0.0203434456037747
443,ZMAT3,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357
444,ZNF639,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357
445,ZNF750,SOMATIC,1.6071012482810222e-08,0.0365630103656301


In [23]:
len(set(hpv_positive_combined_genes['gene_name']))

447

In [24]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [25]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
138,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,BCL6,"11224600, 11420458, 14685876, 17429099",4.0
156,CLDN1,AMPLIFICATION,6.591105366473553e-53,0.0115776022868357,CLDN1,"15170668, 17091452",2.0
164,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0
188,FANCB,DELETION,8.234936784924659e-31,0.0203434456037747,FANCB,17409780,1.0
311,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
336,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,RFC4,16467079,1.0
398,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,SOX2,15942670,1.0
402,STS,DELETION,8.234936784924659e-31,0.0203434456037747,STS,11746990,1.0
409,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,TLR7,17201162,1.0


In [26]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0].reset_index(drop=True)

In [27]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/HPV Positive validated genes.csv')

In [28]:
hpv_positive_genes_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
0,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,BCL6,"11224600, 11420458, 14685876, 17429099",4.0
1,CLDN1,AMPLIFICATION,6.591105366473553e-53,0.0115776022868357,CLDN1,"15170668, 17091452",2.0
2,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0
3,FANCB,DELETION,8.234936784924659e-31,0.0203434456037747,FANCB,17409780,1.0
4,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
5,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,RFC4,16467079,1.0
6,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,SOX2,15942670,1.0
7,STS,DELETION,8.234936784924659e-31,0.0203434456037747,STS,11746990,1.0
8,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,TLR7,17201162,1.0


In [29]:
hpv_positive_connected_genes_with_lit = connect_to_genes(hpv_positive_genes_with_lit)

9it [00:00, 10.66it/s]


In [30]:
hpv_positive_connected_genes_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES,literature_validated_connected_genes,num_literature_validated_connected_genes,PMIDs_connected_genes
0,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,BCL6,"11224600, 11420458, 14685876, 17429099",4.0,"STAT3, STAT1, TP53, CDKN1B, CXCR4, CD5, MYC, J...",22.0,"11244819, 11396137, 11565907, 11595539, 119561..."
1,CLDN1,AMPLIFICATION,6.591105366473553e-53,0.0115776022868357,CLDN1,"15170668, 17091452",2.0,"CTNNB1, EPHA2, CDH1, CD9, CLDN3",5.0,"12494475, 16309192, 18030354, 18425361, 184857..."
2,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0,"BCL3, TP53, NFKB1, TNFRSF1A, CASP8, SMAD7, FAD...",13.0,"18497982, 16387424, 16773191, 16953224, 184979..."
3,FANCB,DELETION,8.234936784924659e-31,0.0203434456037747,FANCB,17409780,1.0,"HES1, MUS81, FANCF, FANCC, FANCD2, FANCG, FANC...",10.0,"14517836, 17321670, 12539049, 11807777, 128106..."
4,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0,"PDGFA, ILK, HCK, SRC, JAK3, ERBB2, PMS2, PTEN,...",66.0,"15543611, 16002527, 16342249, 16630292, 167572..."
5,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,RFC4,16467079,1.0,"LIG1, RAD51AP1, RAD51, ATM, MCM2, BIRC5, MCM4,...",25.0,"11839717, 11872961, 12855631, 12901728, 129499..."
6,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,SOX2,15942670,1.0,"TCF4, GATA4, LIF, PTEN, FGF4, ANXA1, DNMT3B, G...",40.0,"11168856, 11575852, 11798963, 11887001, 121460..."
7,STS,DELETION,8.234936784924659e-31,0.0203434456037747,STS,11746990,1.0,"CYP1A2, CD99, CYP1A1, CYP3A4, CYP3A5, CYP2E1, ...",9.0,"11860825, 12110344, 12359752, 12767509, 128725..."
8,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,TLR7,17201162,1.0,"CD80, IL1B, STAT3, STAT1, CD274, NFKB1, TLR4, ...",18.0,"14960242, 15052669, 15694020, 15876961, 164337..."


In [31]:
all_neighbors = []
for genes in hpv_positive_connected_genes_with_lit['literature_validated_connected_genes']:
    all_neighbors.extend([g.strip() for g in genes.split(',') if g.strip()])

print(len(set(all_neighbors)))

173


In [32]:
hpv_positive_connected_genes_with_lit.to_csv('Results/HPV Positive validated indirect gene results.csv')

In [33]:
hpv_positive_connected_genes_with_lit_explicit = connect_to_genes_explicit(hpv_positive_genes_with_lit)

In [34]:
hpv_positive_connected_genes_with_lit_explicit[hpv_positive_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]

,risk_gene,MUT_TYPE,q_value,empirical_q_value,PMID,NUMBER_OF_ARTICLES,neighbor_gene,neighbor_is_lit_validated,neighbor_PMID,neighbor_NUMBER_OF_ARTICLES,neighbor_PMIDs
2,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,"11224600, 11420458, 14685876, 17429099",4.0,STAT3,True,"11222718, 11426647, 11773428, 12067972, 121624...",41.0,"11222718, 11426647, 11773428, 12067972, 121624..."
6,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,"11224600, 11420458, 14685876, 17429099",4.0,STAT1,True,"11222718, 12162406, 12963127, 15947106, 162491...",8.0,"11222718, 12162406, 12963127, 15947106, 162491..."
11,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,"11224600, 11420458, 14685876, 17429099",4.0,TP53,True,"11390535, 11445847, 11445859, 11916556, 125898...",29.0,"11390535, 11445847, 11445859, 11916556, 125898..."
13,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,"11224600, 11420458, 14685876, 17429099",4.0,CDKN1B,True,"17030811, 18288099",2.0,"17030811, 18288099"
14,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,"11224600, 11420458, 14685876, 17429099",4.0,CXCR4,True,"12160842, 12519884, 12901795, 14567988, 146932...",21.0,"12160842, 12519884, 12901795, 14567988, 146932..."
...,...,...,...,...,...,...,...,...,...,...,...
735,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,17201162,1.0,CD40,True,"11168856, 11675352, 15330155, 15876961, 158787...",9.0,"11168856, 11675352, 15330155, 15876961, 158787..."
739,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,17201162,1.0,CAMP,True,15062572,1.0,15062572
741,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,17201162,1.0,IRF1,True,15947106,1.0,15947106
748,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,17201162,1.0,CXCL10,True,"15761501, 15947106, 18182249",3.0,"15761501, 15947106, 18182249"


In [35]:
hpv_positive_connected_genes_with_lit_explicit = hpv_positive_connected_genes_with_lit_explicit[hpv_positive_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]
hpv_positive_connected_genes_with_lit_explicit.to_csv('Results/HPV Positive validated indirect gene results explicit.csv')

In [36]:
hpv_positive_connected_genes_with_lit_explicit['neighbor_gene'].nunique()

173

## HPV-

#### Genes

In [37]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()


In [38]:
hpv_negative_combined_genes

,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343
1,ABCC5-AS1,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343
2,ABCF3,AMPLIFICATION,1.4051416817717392e-199,0.0043623451388343
3,AC006328.1,DELETION,5.227396253868678e-285,0.0191774659792392
4,AC006328.2,DELETION,5.227396253868678e-285,0.0191774659792392
...,...,...,...,...
885,ZNF835,SOMATIC,1.0196847272849249e-05,0.005960073448722
886,ZNF839P1,DELETION,3.6107825764116486e-286,0.0191774659792392
887,ZNF885P,DELETION,1.4255963792291006e-287,0.0191774659792392
888,ZNF886P,DELETION,1.4255963792291006e-287,0.0191774659792392


In [39]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [40]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0].reset_index(drop=True)

In [41]:
hpv_negative_gene_results_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
0,ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343,ABCC5,18359711,1.0
1,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,BCL6,"11224600, 11420458, 14685876, 17429099",4.0
2,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,CASP8,16857411,1.0
3,CDH7,SOMATIC,0.0040235357729719,0.005960073448722,CDH7,12118382,1.0
4,CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722",CDKN2A,"11309301, 11720478, 12459644, 12632081, 126844...",16.0
5,CDKN2B,DELETION,0.0,0.0191774659792392,CDKN2B,"12632081, 17673925",2.0
6,CLDN1,AMPLIFICATION,4.2419726156021995e-197,0.0043623451388343,CLDN1,"15170668, 17091452",2.0
7,COL11A1,SOMATIC,1.5777323720244649e-15,0.005960073448722,COL11A1,15023835,1.0
8,CSMD3,SOMATIC,5.936981595700065e-36,0.005960073448722,CSMD3,12906867,1.0
9,DCC,SOMATIC,0.0165073700948121,0.005960073448722,DCC,"11603424, 12374684, 16400327",3.0


In [42]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/HPV Negative validated genes.csv')

In [43]:
len(set(hpv_negative_gene_results_with_lit['gene_name']))

31

In [44]:
hpv_negative_connected_genes_with_lit = connect_to_genes(hpv_negative_gene_results_with_lit)

31it [00:03, 10.10it/s]


In [45]:
hpv_negative_connected_genes_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES,literature_validated_connected_genes,num_literature_validated_connected_genes,PMIDs_connected_genes
0,ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343,ABCC5,18359711,1.0,,0.0,
1,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,BCL6,"11224600, 11420458, 14685876, 17429099",4.0,"STAT3, STAT1, TP53, CDKN1B, CXCR4, CD5, MYC, J...",22.0,"11244819, 11396137, 11565907, 11595539, 119561..."
2,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,CASP8,16857411,1.0,"MMP9, SRC, FADD, TNF, BAX, PTEN, BID, XIAP, AT...",39.0,"17722957, 18497982, 11244819, 11396137, 115659..."
3,CDH7,SOMATIC,0.0040235357729719,0.005960073448722,CDH7,12118382,1.0,"CDH11, CDH3, CDH6, CTNNB1",4.0,"15947106, 18490578, 15195113, 16001430, 171314..."
4,CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722",CDKN2A,"11309301, 11720478, 12459644, 12632081, 126844...",16.0,"CDK6, PIK3CA, FHIT, RASSF1, SRC, ERBB2, PTEN, ...",64.0,"17722957, 18497982, 11839717, 11872961, 128556..."
5,CDKN2B,DELETION,0.0,0.0191774659792392,CDKN2B,"12632081, 17673925",2.0,"CDK6, SP1, SMAD3, TP53, CDKN1B, CDKN2A, CDK2, ...",16.0,"12434298, 12475396, 15958520, 16792505, 121615..."
6,CLDN1,AMPLIFICATION,4.2419726156021995e-197,0.0043623451388343,CLDN1,"15170668, 17091452",2.0,"CTNNB1, EPHA2, CDH1, CD9, CLDN3",5.0,"12494475, 16309192, 18030354, 18425361, 184857..."
7,COL11A1,SOMATIC,1.5777323720244649e-15,0.005960073448722,COL11A1,15023835,1.0,"COL4A1, COL5A1, MMP11, COL1A2, COL6A2, DDR2",6.0,"39138492, 17914555, 17786355, 18023033, 126802..."
8,CSMD3,SOMATIC,5.936981595700065e-36,0.005960073448722,CSMD3,12906867,1.0,"ANXA13, LRP1B",2.0,"18000363, 15172977, 16857411, 16918994"
9,DCC,SOMATIC,0.0165073700948121,0.005960073448722,DCC,"11603424, 12374684, 16400327",3.0,"FYN, CASP9, SRC",3.0,"17961551, 12543167, 12771142"


In [46]:
all_neighbors = []
for genes in hpv_negative_connected_genes_with_lit['literature_validated_connected_genes']:
    all_neighbors.extend([g.strip() for g in genes.split(',') if g.strip()])

print(len(set(all_neighbors)))

373


In [47]:
hpv_negative_connected_genes_with_lit.to_csv('Results/HPV Negative validated indirect gene results.csv')

In [48]:
hpv_negative_connected_genes_with_lit_explicit = connect_to_genes_explicit(hpv_negative_connected_genes_with_lit)

In [49]:
hpv_negative_connected_genes_with_lit_explicit[hpv_negative_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]

,risk_gene,MUT_TYPE,q_value,empirical_q_value,PMID,NUMBER_OF_ARTICLES,neighbor_gene,neighbor_is_lit_validated,neighbor_PMID,neighbor_NUMBER_OF_ARTICLES,neighbor_PMIDs
3,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4.0,STAT3,True,"11222718, 11426647, 11773428, 12067972, 121624...",41.0,"11222718, 11426647, 11773428, 12067972, 121624..."
7,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4.0,STAT1,True,"11222718, 12162406, 12963127, 15947106, 162491...",8.0,"11222718, 12162406, 12963127, 15947106, 162491..."
12,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4.0,TP53,True,"11390535, 11445847, 11445859, 11916556, 125898...",29.0,"11390535, 11445847, 11445859, 11916556, 125898..."
14,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4.0,CDKN1B,True,"17030811, 18288099",2.0,"17030811, 18288099"
15,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"11224600, 11420458, 14685876, 17429099",4.0,CXCR4,True,"12160842, 12519884, 12901795, 14567988, 146932...",21.0,"12160842, 12519884, 12901795, 14567988, 146932..."
...,...,...,...,...,...,...,...,...,...,...,...
2967,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898...",29.0,POLK,True,17494052,1.0,17494052
2976,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898...",29.0,E2F1,True,"11796486, 12036945, 12966732, 14696419, 151189...",6.0,"11796486, 12036945, 12966732, 14696419, 151189..."
2978,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898...",29.0,MSH6,True,15452164,1.0,15452164
2984,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898...",29.0,E2F2,True,14684733,1.0,14684733


In [50]:
hpv_negative_connected_genes_with_lit_explicit[hpv_negative_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]['neighbor_gene'].nunique()

373

In [51]:
hpv_negative_connected_genes_with_lit_explicit = hpv_negative_connected_genes_with_lit_explicit[hpv_negative_connected_genes_with_lit_explicit['neighbor_is_lit_validated']]
hpv_negative_connected_genes_with_lit_explicit.to_csv('Results/HPV Negative validated indirect gene results explicit.csv')

In [52]:
print(f"Number of unique genes in hpv pos genes with lit not in hpv neg: {len(set(hpv_positive_genes_with_lit['gene_name']) - set(hpv_negative_gene_results_with_lit['gene_name']))}")
print(f"genes unique to hpv pos: {set(hpv_positive_genes_with_lit['gene_name']) - set(hpv_negative_gene_results_with_lit['gene_name'])}")
print(f"Number of unique genes in hpv neg genes with lit not in hpv pos: {len(set(hpv_negative_gene_results_with_lit['gene_name']) - set(hpv_positive_genes_with_lit['gene_name']))}")
print(f"genes unique to hpv neg: {set(hpv_negative_gene_results_with_lit['gene_name']) - set(hpv_positive_genes_with_lit['gene_name'])}")

print(f"Number of unique genes in both hpv pos and hpv neg genes with lit: {len(set(hpv_positive_genes_with_lit['gene_name']).intersection(set(hpv_negative_gene_results_with_lit['gene_name'])))}")
print(f"genes in both hpv pos and hpv neg: {set(hpv_positive_genes_with_lit['gene_name']).intersection(set(hpv_negative_gene_results_with_lit['gene_name']))}")

print(f"Number of unique genes across both hpv pos and hpv neg genes with lit: {len(set(hpv_positive_genes_with_lit['gene_name']).union(set(hpv_negative_gene_results_with_lit['gene_name'])))}")

Number of unique genes in hpv pos genes with lit not in hpv neg: 4
genes unique to hpv pos: {'STS', 'CYLD', 'FANCB', 'TLR7'}
Number of unique genes in hpv neg genes with lit not in hpv pos: 26
genes unique to hpv neg: {'PRKCI', 'DCC', 'HLA-A', 'NOTCH1', 'TERC', 'TP53', 'EPHA2', 'CDKN2A', 'PDCD10', 'CASP8', 'EIF4G1', 'HRAS', 'LRP1B', 'REG1A', 'CDH7', 'RHOA', 'KEAP1', 'SELP', 'COL11A1', 'DVL3', 'RAC1', 'ABCC5', 'CDKN2B', 'MAGEC1', 'MYNN', 'CSMD3'}
Number of unique genes in both hpv pos and hpv neg genes with lit: 5
genes in both hpv pos and hpv neg: {'CLDN1', 'PIK3CA', 'SOX2', 'RFC4', 'BCL6'}
Number of unique genes across both hpv pos and hpv neg genes with lit: 35


### sanity check of the results

In [53]:
protein_interaction[(protein_interaction['Translated_protein_1'] == 'TP53') & 
                    (protein_interaction['Translated_protein_2'] == 'THRB')]

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
2694854,9606.ENSP00000269305,9606.ENSP00000379904,705,TP53,THRB
